## Processo de ETL na Arquitetuta Medalhao (Camadas  Raw > Bronze > Silver > Gold)

In [1]:
# inicializa o SparkSesion
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import when, col, count, avg, round

# Criando App Spark
spark = SparkSession \
    .builder \
    .appName('ETL Salarios Profissionais de TI') \
    .getOrCreate()

### Camada Raw
Carregando e lendo fonte de dados

In [2]:
df_raw = spark.read.csv("../source/ti_salaries.csv", header=True, inferSchema=True)     
df_raw.show(5)                            

+---------+----------------+---------------+------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|         job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|     2025|              SE|             FT|Solutions Engineer|214000|            USD|       214000|                US|         100|              US|           M|
|     2025|              SE|             FT|Solutions Engineer|136000|            USD|       136000|                US|         100|              US|           M|
|     2025|              MI|             FT|     Data Engineer|158800|            USD|       158800|                AU|           0|              AU|           M|
|     2025|           

## Camada Bronze
Salvando df_raw na Camada Bronze no processamento


In [3]:
df_bronze = df_raw

In [4]:
print("Antes de padronizar as colunas English para Português na Bronze:")
df_bronze.count()

Antes de padronizar as colunas English para Português na Bronze:


133349

## Camada Silver
Salvando df_bronze como df_silver no processamento

In [5]:
df_silver = df_bronze

## Padronizando colunas do DF para Portugues

In [6]:
# Dicionário de tradução
colunas_traduzidas = {
    'work_year': 'ano',
    'experience_level': 'senioridade',
    'employment_type': 'contrato',
    'job_title': 'cargo',
    'salary': 'salario_bruto',
    'salary_currency': 'moeda',
    'salary_in_usd': 'salary_usd',
    'employee_residence': 'residencia_profissional',
    'remote_ratio': 'trab_remoto',
    'company_location': 'empresa',
    'company_size': 'tamanho_empresa'
}

# Renomeando cols
for old_col, new_col in colunas_traduzidas.items():
  df_silver = df_silver.withColumnRenamed(old_col, new_col)


## Amostra do Antes e Depois - Titulos das Colunas

In [7]:
print("Antes de padronizar as colunas English para Português na Bronze:")
df_bronze.printSchema()

print("Depois de padronizar as colunas English para Português na Silver:")
df_silver.printSchema()

Antes de padronizar as colunas English para Português na Bronze:
root
 |-- work_year: integer (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- remote_ratio: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)

Depois de padronizar as colunas English para Português na Silver:
root
 |-- ano: integer (nullable = true)
 |-- senioridade: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- salario_bruto: integer (nullable = true)
 |-- moeda: string (nullable = true)
 |-- salary_usd: integer (nullable = true)
 |-- residencia_profissional: string (nullable = true)
 |-- trab_remoto: intege

## Removendo 10 registros da col Ano

In [8]:
df_silver = df_silver.filter(col("ano").isNotNull())
print("Total de linhas depois:", df_silver.count())

Total de linhas depois: 133339


## Amostra do Antes e Depois das Remoção de 10 Linhas da cols ano

In [9]:
print("Antes de Remover 10 linhas da coluna ano na camada Bronze")
print(f"Bronze: {df_bronze.count()} registros")

print("\nDepois de Remover 10 linhas da coluna ano na camada Silver")
print(f"Silver: {df_silver.count()} registros")

Antes de Remover 10 linhas da coluna ano na camada Bronze
Bronze: 133349 registros

Depois de Remover 10 linhas da coluna ano na camada Silver
Silver: 133339 registros


## Avaliando variaveis categoricas
Nesse base de dados temos varias variaveis categoricas que vou ter que traduzir para o nome por extenso para entendimento dessas variaveis

EX: A coluna **tamanho_empresa**, refere ao tamanho da empresa como: 
* S-Pequeno, 
* M-Media e 
* G-Grande

O mesmo serve para as outras variaveis categoricas, essas traduções foram extraidas do dicionário da base de dados.

### Analisando da categoria: tamanho_empresa

In [10]:
df_silver.groupBy('tamanho_empresa').count().orderBy(col('count')).show()

+---------------+------+
|tamanho_empresa| count|
+---------------+------+
|              S|   214|
|              L|  3571|
|              M|129554|
+---------------+------+



### Padronizando Coluna tamanho_empresa - Novo DF com colunas transformadas - Saida

In [11]:
df_silver = df_silver.withColumn(
    'tamanho_empresa',
    when(col('tamanho_empresa') == 'M', 'Media')
    .when(col('tamanho_empresa') == 'L', 'Grande')
    .when(col('tamanho_empresa') == 'S', 'Pequena')
    .otherwise(col('tamanho_empresa') )
)
df_silver.groupBy('tamanho_empresa').count().show()

+---------------+------+
|tamanho_empresa| count|
+---------------+------+
|          Media|129554|
|        Pequena|   214|
|         Grande|  3571|
+---------------+------+



### Análise categoria Senioridade

In [12]:
# Contagem de valores distintos
df_silver.groupBy('senioridade').count().orderBy(col('count')).show()

+-----------+-----+
|senioridade|count|
+-----------+-----+
|         EX| 3200|
|         EN|12441|
|         MI|40462|
|         SE|77236|
+-----------+-----+



### Padronizando Coluna Senioridade - Novo DF com colunas transformadas - Saida

In [13]:
df_silver = df_silver.withColumn(
    'senioridade',
    when(col('senioridade') == 'EX', "Executivo")
    .when(col('senioridade') == 'MI', "Pleno")
    .when(col('senioridade') == 'EN', "Júnior")
    .when(col('senioridade') == 'SE', "Sênior")
    .otherwise(col('senioridade') )
)
df_silver.groupBy('senioridade').count().show()

+-----------+-----+
|senioridade|count|
+-----------+-----+
|  Executivo| 3200|
|     Júnior|12441|
|     Sênior|77236|
|      Pleno|40462|
+-----------+-----+



### Análise categoria Contrato

In [14]:
df_silver.groupBy("contrato").count().show()

+--------+------+
|contrato| count|
+--------+------+
|      FT|132553|
|      PT|   376|
|      CT|   394|
|      FL|    16|
+--------+------+



### Padronizando  Coluna col Contrato - Novo DF transformadas - Saida

In [15]:
df_silver = df_silver.withColumn("contrato",
    when(col("contrato") == "FT", "Tempo_Integral")
    .when(col("contrato") == "PT", "Parcial")
    .when(col("contrato") == "CP", "Contrato")
    .when(col("contrato") == "FL", "Freelancer")
    .otherwise(col("contrato"))
)
df_silver.groupBy("contrato").count().show()

+--------------+------+
|      contrato| count|
+--------------+------+
|            CT|   394|
|Tempo_Integral|132553|
|       Parcial|   376|
|    Freelancer|    16|
+--------------+------+



### Análise da Coluna trab_remoto


In [16]:
df_silver.groupBy('trab_remoto').count().show()

+-----------+------+
|trab_remoto| count|
+-----------+------+
|        100| 27716|
|         50|   318|
|          0|105305|
+-----------+------+



## 🔄 Conversão da coluna `trab_remoto` de `int` para `string`

A coluna `trab_remoto` é originalmente armazenada como um valor numérico do tipo **`int`**, representando o percentual de trabalho remoto.

Antes de realizar a padronização dos valores, a coluna é convertida para **`string`**, permitindo tratar os códigos como categorias.

Os valores são então convertidos conforme o mapeamento:

- `0` → **Presencial**
- `50` → **Híbrido**
- `100` → **Remoto**

Essa conversão separa a etapa de **adequação do tipo de dado** da etapa de **padronização dos valores categóricos**, tornando o processo de transformação mais explícito e consistente.


In [17]:
# Convertendo coluna "Remoto" para String(int)
df_silver = df_silver.withColumn("trab_remoto", col("trab_remoto").cast("string"))

### Padronizando coluna trab_remoto - Saida novo DF

In [18]:
df_silver = df_silver.withColumn(
    'trab_remoto',
    when(col('trab_remoto') == '0', "Presencial")
    .when(col('trab_remoto') == '50', "Hibrido")
    .when(col('trab_remoto') == '100', "Remoto")
    .otherwise(col('trab_remoto') )
)

print('Valores atualizados na col "trab_remoto"')
df_silver.groupBy('trab_remoto').count().show(5)

Valores atualizados na col "trab_remoto"
+-----------+------+
|trab_remoto| count|
+-----------+------+
| Presencial|105305|
|     Remoto| 27716|
|    Hibrido|   318|
+-----------+------+



## Historico de Alterações das Cols Categorica do DF

In [19]:
print("DF Antes das Colunas Transformadas - Valores distintos")
df_bronze.select("experience_level","employment_type","remote_ratio","company_size").distinct().show(5)

print("DF Depois das Colunas Transformadas - Valores distintos")
df_silver.select("senioridade","contrato","trab_remoto","tamanho_empresa").distinct().show(5)

DF Antes das Colunas Transformadas - Valores distintos
+----------------+---------------+------------+------------+
|experience_level|employment_type|remote_ratio|company_size|
+----------------+---------------+------------+------------+
|              MI|             PT|         100|           M|
|              MI|             FT|           0|           M|
|              EX|             FT|         100|           M|
|              SE|             FT|          50|           S|
|              EX|             FT|           0|           S|
+----------------+---------------+------------+------------+
only showing top 5 rows

DF Depois das Colunas Transformadas - Valores distintos
+-----------+--------------+-----------+---------------+
|senioridade|      contrato|trab_remoto|tamanho_empresa|
+-----------+--------------+-----------+---------------+
|     Júnior|            CT| Presencial|          Media|
|     Júnior|Tempo_Integral| Presencial|         Grande|
|     Sênior|Tempo_Integral|  

### Validando Amostra Dados Final do ETL no DataFrame 

Valida todas as alterações realizadas no DF

In [20]:
print("Qtde de Linhas e colunas do gold_analytics", df_silver.count(), len(df_silver.columns) )
print("Tipos de dados da Cols") 
df_silver.printSchema() 
print("Valida leitura de dados do DF Final"), 
df_silver.show(5) 

Qtde de Linhas e colunas do gold_analytics 133339 11
Tipos de dados da Cols
root
 |-- ano: integer (nullable = true)
 |-- senioridade: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- salario_bruto: integer (nullable = true)
 |-- moeda: string (nullable = true)
 |-- salary_usd: integer (nullable = true)
 |-- residencia_profissional: string (nullable = true)
 |-- trab_remoto: string (nullable = true)
 |-- empresa: string (nullable = true)
 |-- tamanho_empresa: string (nullable = true)

Valida leitura de dados do DF Final
+----+-----------+--------------+------------------+-------------+-----+----------+-----------------------+-----------+-------+---------------+
| ano|senioridade|      contrato|             cargo|salario_bruto|moeda|salary_usd|residencia_profissional|trab_remoto|empresa|tamanho_empresa|
+----+-----------+--------------+------------------+-------------+-----+----------+-----------------------+-----------+-------+-

### Salvando ETL Final na Camada Gold

In [21]:
df_silver \
    .coalesce(1) \
    .write.parquet("../gold_analytics", mode="overwrite")

### Automatizando para renomear nome do arquivo

Objetivo não precisar pegar o nome do arquivo gerado de forma aleatória pelo Spark

In [22]:
import os
import shutil

# caminho da pasta
output_dir = "../gold_analytics"

# Procura arq gerado pelo spark
for file in os.listdir(output_dir):
    if file.startswith("part-") and file.endswith(".parquet"):
        old_path = os.path.join(output_dir, file)
        new_path = os.path.join(output_dir, "df_gold_analytics.parquet")
        shutil.move(old_path, new_path)
        print(f"Arq renomeado para:{new_path}")
        break

Arq renomeado para:../gold_analytics\df_gold_analytics.parquet


### Leitura e Validacao do df_gold_analytics
Depois de persistido na camada gold, pode ocorrer alterações por isso uma **segunda checagem**

In [24]:
df_analytics = spark.read.parquet("../gold_analytics/df_gold_analytics.parquet")
print("Qtde de Linhas e colunas do gold_analytics") 
print(df_analytics.count(), len(df_analytics.columns) )
print("Schema de dados do gold_analytics")
print(df_analytics.printSchema())
print("Valida leitura de dados do DF gold_analytics")
print(df_analytics.show(5))

Qtde de Linhas e colunas do gold_analytics
133339 11
Schema de dados do gold_analytics
root
 |-- ano: integer (nullable = true)
 |-- senioridade: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- salario_bruto: integer (nullable = true)
 |-- moeda: string (nullable = true)
 |-- salary_usd: integer (nullable = true)
 |-- residencia_profissional: string (nullable = true)
 |-- trab_remoto: string (nullable = true)
 |-- empresa: string (nullable = true)
 |-- tamanho_empresa: string (nullable = true)

None
Valida leitura de dados do DF gold_analytics
+----+-----------+--------------+------------------+-------------+-----+----------+-----------------------+-----------+-------+---------------+
| ano|senioridade|      contrato|             cargo|salario_bruto|moeda|salary_usd|residencia_profissional|trab_remoto|empresa|tamanho_empresa|
+----+-----------+--------------+------------------+-------------+-----+----------+--------------------

## Fim